# TaskAug on PTB-XL — Experiment Runner
**DL4H Spring 2026 | Paul Garcia · Rogelio Medina · Cesar Nava**

Reproduces Raghu et al. (CHIL 2022) TaskAug on PTB-XL (4 tasks: MI, HYP, STTC, CD).

Works **locally** (CPU/GPU) and on **Google Colab** (T4).  
PTB-XL is downloaded from **Kaggle** (faster + more stable than PhysioNet).

---
### Experiment matrix
| Method | Tasks | N |
|---|---|---|
| No augmentation (baseline) | MI, HYP, STTC, CD | 1 000, 5 000 |
| TaskAug — frozen policy (ablation) | MI, HYP, STTC, CD | 1 000 |
| TaskAug — global magnitudes (ablation) | MI, HYP, STTC, CD | 1 000 |
| TaskAug — full | MI, HYP, STTC, CD | 1 000, 5 000 |

Results are saved incrementally as JSON → safe to interrupt and resume.

## 0. Hardware check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Running on CPU — training will be slow; consider a GPU runtime.')

## 1. Repo + dependencies

**Local:** set `REPO_ROOT` to the absolute path of your `pyhealth-dl4h-sp26` clone.  
**Colab:** leave as-is; the repo is cloned automatically to `/content/pyhealth-dl4h-sp26`.

In [ ]:
import os, sys, subprocess

# ── Configure: set these for your environment ─────────────────────────────
IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if IN_COLAB:
    REPO_ROOT = '/content/pyhealth-dl4h-sp26'
else:
    # ← Change to the absolute path of your local clone
    REPO_ROOT = os.path.expanduser('~/Developer/MCS/Semester 2/CS598_DeepLearningForHealthcare/pyhealth-dl4h-sp26')

print('IN_COLAB :', IN_COLAB)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
# ── Clone / pull repo ─────────────────────────────────────────────────────
if IN_COLAB:
    if not os.path.isdir(REPO_ROOT):
        !git clone -b feature/pg/taskaug-ecg \
            https://github.com/paulgarciaro/pyhealth-dl4h-sp26.git {REPO_ROOT}
    else:
        print('Repo present, pulling latest ...')
        subprocess.run(['git', '-C', REPO_ROOT, 'pull', '--ff-only'], check=True)
else:
    # Local: just pull latest from origin
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '--ff-only'], check=True)

# Add repo to sys.path (editable install — no pip install -e needed)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Verify key files
for f in ['pyhealth/datasets/ptbxl.py',
          'pyhealth/tasks/ecg_classification_ptbxl.py',
          'pyhealth/models/taskaug_resnet.py',
          'experiments/colab_setup.py']:
    assert os.path.isfile(f'{REPO_ROOT}/{f}'), f'Missing: {f}'
print('Repo verified ✓')

In [ ]:
# ── Install runtime dependencies ──────────────────────────────────────────
# (Skip packages already present to avoid re-downloading on every run)
!pip install wfdb scipy scikit-learn pandas tqdm polars pyarrow pydantic \
             platformdirs mne litdata filelock dask distributed narwhals \
             more-itertools einops kaggle -q
print('Dependencies ready ✓')

## 2. PTB-XL dataset

### 2a. Kaggle credentials
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **Create New Token** → downloads `kaggle.json`
2. **Local:** place it at `~/.kaggle/kaggle.json` and run `chmod 600 ~/.kaggle/kaggle.json`  
   **Colab:** upload it via the cell below (it will be placed automatically)

In [ ]:
import os
from pathlib import Path

KAGGLE_JSON = Path.home() / '.kaggle' / 'kaggle.json'

if not KAGGLE_JSON.exists():
    if IN_COLAB:
        print('Upload your kaggle.json in the next cell.')
    else:
        print(f'Missing: {KAGGLE_JSON}')
        print('Get it at https://www.kaggle.com/settings → Create New Token')
        print('Then: chmod 600 ~/.kaggle/kaggle.json')
else:
    os.chmod(KAGGLE_JSON, 0o600)
    print('Kaggle credentials found ✓')

In [ ]:
# ── Colab only: upload kaggle.json ────────────────────────────────────────
# Skip this cell if running locally or credentials already exist.
if IN_COLAB and not KAGGLE_JSON.exists():
    from google.colab import files
    print('Select your kaggle.json file to upload:')
    uploaded = files.upload()
    KAGGLE_JSON.parent.mkdir(exist_ok=True)
    KAGGLE_JSON.write_bytes(list(uploaded.values())[0])
    os.chmod(KAGGLE_JSON, 0o600)
    print('kaggle.json saved ✓')
else:
    print('Skipping upload (credentials already present or running locally).')

### 2b. Configure paths

In [ ]:
if IN_COLAB:
    # Mount Drive so PTB-XL and results survive session restarts
    from google.colab import drive
    drive.mount('/content/drive')
    PTB_XL_ROOT = '/content/drive/MyDrive/DL4H_SP26/ptb-xl-1.0.3'
    RESULTS_DIR = '/content/drive/MyDrive/DL4H_SP26/results'
else:
    # ← Change these to your local paths
    PTB_XL_ROOT = os.path.expanduser('~/data/ptb-xl-1.0.3')
    RESULTS_DIR = f'{REPO_ROOT}/experiments/results'

os.makedirs(PTB_XL_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('PTB-XL root:', PTB_XL_ROOT)
print('Results dir:', RESULTS_DIR)

### 2c. Download from Kaggle
Kaggle is significantly faster and more stable than PhysioNet for large files.  
The cell is **idempotent** — it skips the download if the data is already present.

In [ ]:
import subprocess
from pathlib import Path

def _ptbxl_present(root):
    p = Path(root)
    return (p / 'ptbxl_database.csv').is_file() and (p / 'records500').is_dir()

KAGGLE_SLUGS = [
    'khyeh0719/ptb-xl-dataset',
    'bjoernjostein/ptb-xl-electrocardiography-dataset',
]

if _ptbxl_present(PTB_XL_ROOT):
    print('PTB-XL already present, skipping download.')
else:
    downloaded = False
    for slug in KAGGLE_SLUGS:
        print(f'Trying: kaggle datasets download -d {slug} ...')
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', slug,
             '-p', PTB_XL_ROOT, '--unzip'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print('Download complete ✓')
            downloaded = True
            break
        else:
            print(f'  Failed: {result.stderr.strip()}')

    if not downloaded:
        print('All Kaggle slugs failed. Falling back to PhysioNet wget ...')
        !wget -q -P {PTB_XL_ROOT} https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
        !wget -q -P {PTB_XL_ROOT} https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv
        !wget -q -r -nH --cut-dirs=3 -P {PTB_XL_ROOT} \
            https://physionet.org/files/ptb-xl/1.0.3/records500/

# Verify
assert _ptbxl_present(PTB_XL_ROOT), f'PTB-XL files not found at {PTB_XL_ROOT}'
print('PTB-XL verified ✓')

## 3. Import firewall + shared utilities
All the import-chain fixes and experiment functions live in `experiments/colab_setup.py`.  
The cell below pulls the latest version from git and executes it — no notebook reload needed.

In [ ]:
import subprocess, sys, os

# Always pull the latest colab_setup.py before executing
subprocess.run(['git', '-C', REPO_ROOT, 'pull', '--ff-only'], check=True)

# Executes colab_setup.py in the current namespace, making available:
#   DEVICE, TASKS, N_SIZES, N_SPLITS, EPOCHS, BATCH
#   PTBXLDataset, ECGBinaryClassificationPTBXL, binary_metrics_fn
#   get_dataloader, split_by_patient
#   ResNet1D, TaskAugPolicy, BiLevelTrainer
#   build_loaders(), run_split()
exec(open(f'{REPO_ROOT}/experiments/colab_setup.py').read())

# Build ptbxl-pyhealth.csv if not present
if not os.path.isfile(f'{PTB_XL_ROOT}/ptbxl-pyhealth.csv'):
    print('Building ptbxl-pyhealth.csv ...')
    PTBXLDataset.prepare_metadata(PTB_XL_ROOT)
print('Metadata ready ✓')

## 4. Experiment runner

In [ ]:
import json, time
import numpy as np

def run_experiment(method, tasks=TASKS, n_sizes=N_SIZES,
                   n_splits=N_SPLITS, ptb_root=PTB_XL_ROOT,
                   results_dir=RESULTS_DIR):
    """Run method across all tasks × N × splits. JSON results saved incrementally."""
    os.makedirs(results_dir, exist_ok=True)
    all_results = {}

    for task in tasks:
        for n in n_sizes:
            # Ablations only at N=1000 (matches paper Table 3)
            if method in ('frozen_policy', 'global_mag') and n != 1000:
                continue

            key = f'{method}__{task}__n{n}'
            result_file = f'{results_dir}/{key}.json'

            if os.path.isfile(result_file):
                print(f'[SKIP] {key} — already done')
                with open(result_file) as f:
                    all_results[key] = json.load(f)
                continue

            print(f'\n>>> {key}  (device={DEVICE})')
            split_scores = []
            for seed in range(n_splits):
                t0 = time.time()
                scores = run_split(task, n, seed, method, ptb_root,
                                   f'{results_dir}/ckpts')
                elapsed = time.time() - t0
                split_scores.append(scores)
                print(f'  split {seed:02d} | '
                      f'AUROC={scores["roc_auc"]:.4f}  '
                      f'AUPRC={scores["pr_auc"]:.4f} | {elapsed:.0f}s')

            agg = {
                'roc_auc_mean': float(np.mean([s['roc_auc'] for s in split_scores])),
                'roc_auc_std':  float(np.std( [s['roc_auc'] for s in split_scores])),
                'pr_auc_mean':  float(np.mean([s['pr_auc']  for s in split_scores])),
                'pr_auc_std':   float(np.std( [s['pr_auc']  for s in split_scores])),
                'splits': split_scores,
            }
            all_results[key] = agg
            with open(result_file, 'w') as f:
                json.dump(agg, f, indent=2)
            print(f'  → AUROC {agg["roc_auc_mean"]:.4f} ± {agg["roc_auc_std"]:.4f}'
                  f' | AUPRC {agg["pr_auc_mean"]:.4f} ± {agg["pr_auc_std"]:.4f}')

    return all_results

print('run_experiment() ready ✓')

### 4a. No augmentation baseline

In [ ]:
results_no_aug = run_experiment('no_aug')

### 4b. Frozen policy (ablation)

In [ ]:
results_frozen = run_experiment('frozen_policy', n_sizes=[1000])

### 4c. Global magnitudes (ablation)

In [ ]:
results_global = run_experiment('global_mag', n_sizes=[1000])

### 4d. TaskAug — full method

In [ ]:
results_taskaug = run_experiment('taskaug')

## 5. Results table

In [ ]:
import pandas as pd

def load_result(method, task, n, results_dir=RESULTS_DIR):
    path = f'{results_dir}/{method}__{task}__n{n}.json'
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        return json.load(f)

def fmt(r, metric='roc_auc'):
    if r is None:
        return 'TBD'
    return f"{r[f'{metric}_mean']:.3f} \u00b1 {r[f'{metric}_std']:.3f}"

METHODS = [
    ('no_aug',        'No augmentation'),
    ('frozen_policy', 'TaskAug frozen policy'),
    ('global_mag',    'TaskAug global magnitudes'),
    ('taskaug',       'TaskAug full'),
]

for n in N_SIZES:
    for metric in ('roc_auc', 'pr_auc'):
        label = 'AUROC' if metric == 'roc_auc' else 'AUPRC'
        print(f'\n=== {label}  (N={n}) ===')
        rows = []
        for mk, ml in METHODS:
            if mk in ('frozen_policy', 'global_mag') and n != 1000:
                continue
            row = {'Method': ml}
            for task in TASKS:
                row[task] = fmt(load_result(mk, task, n), metric)
            rows.append(row)
        print(pd.DataFrame(rows).to_string(index=False))

## 6. Save summary JSON

In [ ]:
summary = {}
for mk, _ in METHODS:
    for task in TASKS:
        for n in N_SIZES:
            r = load_result(mk, task, n)
            if r:
                summary[f'{mk}__{task}__n{n}'] = {
                    'roc_auc': f"{r['roc_auc_mean']:.4f}",
                    'pr_auc':  f"{r['pr_auc_mean']:.4f}",
                }

summary_path = f'{RESULTS_DIR}/all_results_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Summary saved → {summary_path}')
print(json.dumps(summary, indent=2))